In [ ]:
################################################
estrai tabella per grafici di serie storiche
################################################

In [ ]:
####################### old versions

In [2]:


import re
import json
import pandas as pd

def estrai_tabella_da_html(percorso_file_html):
    """
    Legge un file HTML di Plotly, estrae la serie di dati e 
    restituisce un DataFrame Pandas strutturato con TOTALI.
    """
    with open(percorso_file_html, 'r', encoding='utf-8') as file:
        html_content = file.read()

    # 1. Troviamo la matrice di dati
    pattern = r'Plotly\.newPlot\(\s*"[^"]+",\s*(\[.*?\])\s*,\s*\{"template"'
    match = re.search(pattern, html_content, re.DOTALL)

    if not match:
        raise ValueError("Non sono riuscito a trovare la matrice dati nell'HTML.")

    json_data = match.group(1)
    traces = json.loads(json_data)

    # 2. Spacchettiamo i dati
    records = []
    for trace in traces:
        nome_cpi = trace.get('name', 'Sconosciuto')
        anni = trace.get('x', [])
        valori = trace.get('y', [])
        
        for anno, valore in zip(anni, valori):
            records.append({
                'Anno': anno,
                'CPI': nome_cpi,
                'COB': valore
            })

    df_grezzo = pd.DataFrame(records)

    # 3. Forgiamo la Tabella Pivot CON I TOTALI (margins=True)
    df_pivot = pd.pivot_table(
        df_grezzo, 
        index='Anno', 
        columns='CPI', 
        values='COB', 
        aggfunc='sum',     # Somma i valori
        margins=True,      # Attiva i totali di riga e colonna
        margins_name='TOTALE' # Etichetta per la riga/colonna dei totali
    ).reset_index()
    
    # 4. Pulizia estetica (rimuove il nome tecnico dell'asse colonne)
    df_pivot.columns.name = None 
    
    return df_pivot


# --- ESEMPIO DI UTILIZZO ---
# Salva il tuo codice html in un file in percorso_file_html e lancia lo script:

percorso_file_html=  "C:\\Users\\lorid\\quarto_report_somministrazione_OML34-2025\\esportazioni_quarto\\grafico_10_COB_CPI_top25Qualif_1525.html"
df_risultato = estrai_tabella_da_html(percorso_file_html)
print(df_risultato)

      Anno  CPI BORMIO  CPI CHIAVENNA  CPI MORBEGNO  CPI SONDRIO  CPI TIRANO  \
0     2015       17674           6893         10693        11105        6425   
1     2016       17528           6078         10204        10413        6900   
2     2017       21901           6297         13353        12145        8828   
3     2018       23693           6297         14889        13985        9933   
4     2019       23799           6270         14849        13366        9287   
5     2020       18821           5462         11908        10956        7806   
6     2021       22515           5932         13491        12157        8187   
7     2022       26509           6359         15225        13784        9275   
8     2023       24847           6042         15666        13038        9469   
9     2024       25770           6348         15182        13252        9397   
10    2025       26203           6225         13229        12496        9432   
11  TOTALE      249260          68203   

In [4]:
#secondo esempio

percorso_file_html="..\\esportazioni_quarto\\grafico_7_COB_NAZ_tutti_contratti_1525_tutte_le_naz.html"

df_risultato = estrai_tabella_da_html(percorso_file_html)
df_risultato

,Anno,AFGHANISTAN,ALBANIA,ALGERIA,ANGOLA,ANTIGUA E BARBUDA,ARABIA SAUDITA,ARGENTINA,ARMENIA,AUSTRALIA,...,UNGHERIA,UNIONE REPUBBLICHE SOCIALISTE SOVIETICHE,URUGUAY,UZBEKISTAN,VENEZUELA,VIETNAM,YEMEN,ZAMBIA,ZIMBABWE,TOTALE
0,2015,6.0,1493.0,16.0,NaN,NaN,5.0,168.0,6.0,23.0,...,103.0,12.0,30.0,NaN,72.0,3.0,NaN,NaN,NaN,80532
1,2016,18.0,1424.0,18.0,1.0,NaN,6.0,118.0,3.0,21.0,...,90.0,43.0,40.0,2.0,62.0,4.0,NaN,NaN,NaN,76145
2,2017,43.0,1601.0,24.0,7.0,NaN,10.0,155.0,21.0,14.0,...,61.0,38.0,48.0,15.0,78.0,6.0,NaN,6.0,NaN,92216
3,2018,21.0,1780.0,35.0,6.0,2.0,8.0,188.0,19.0,36.0,...,55.0,66.0,51.0,7.0,123.0,3.0,NaN,3.0,NaN,100384
4,2019,50.0,2001.0,28.0,10.0,NaN,6.0,197.0,22.0,32.0,...,56.0,53.0,38.0,19.0,171.0,2.0,NaN,NaN,NaN,97645
5,2020,43.0,1773.0,20.0,7.0,NaN,7.0,157.0,8.0,31.0,...,56.0,25.0,17.0,12.0,141.0,1.0,NaN,2.0,NaN,81845
6,2021,37.0,1928.0,33.0,11.0,NaN,7.0,253.0,7.0,28.0,...,60.0,24.0,26.0,7.0,142.0,3.0,NaN,NaN,NaN,93015
7,2022,47.0,2026.0,52.0,8.0,6.0,12.0,448.0,21.0,41.0,...,73.0,18.0,41.0,13.0,224.0,7.0,3.0,2.0,NaN,105284
8,2023,31.0,1693.0,49.0,11.0,7.0,5.0,684.0,16.0,53.0,...,67.0,41.0,40.0,22.0,215.0,10.0,NaN,3.0,5.0,103171
9,2024,33.0,1670.0,33.0,15.0,9.0,7.0,1125.0,16.0,31.0,...,57.0,37.0,35.0,20.0,261.0,10.0,NaN,NaN,2.0,105420


In [ ]:
###################  versione migliorata

In [5]:
import re
import json
import pandas as pd

def estrai_tabella_da_html(percorso_file_html):
    """
    Legge un file HTML di Plotly, estrae la serie di dati e 
    restituisce un DataFrame Pandas strutturato a Crosstab con TOTALI.
    Affina l'estetica rimuovendo i '.0' e i 'NaN'.
    Adattata per gestire dinamicamente qualsiasi Categoria (CPI, Età, Genere, ecc.).
    """
    with open(percorso_file_html, 'r', encoding='utf-8') as file:
        html_content = file.read()

    # 1. Troviamo la matrice di dati
    pattern = r'Plotly\.newPlot\(\s*"[^"]+",\s*(\[.*?\])\s*,\s*\{"template"'
    match = re.search(pattern, html_content, re.DOTALL)

    if not match:
        raise ValueError("Non sono riuscito a trovare la matrice dati nell'HTML.")

    json_data = match.group(1)
    traces = json.loads(json_data)

    # 2. Spacchettiamo i dati usando nomenclature universali
    records = []
    for trace in traces:
        # Usiamo 'nome_categoria' al posto di 'nome_cpi'
        nome_categoria = trace.get('name', 'Sconosciuto')
        anni = trace.get('x', [])
        valori = trace.get('y', [])
        
        for anno, valore in zip(anni, valori):
            records.append({
                'Anno': anno,
                'Categoria': nome_categoria, # Nome colonna generico e flessibile
                'COB': valore
            })

    df_grezzo = pd.DataFrame(records)

    # 3. Forgiamo la Tabella Pivot (Gestendo il caso in cui il dataframe sia vuoto)
    if not df_grezzo.empty:
        df_pivot = pd.pivot_table(
            df_grezzo, 
            index='Anno', 
            columns='Categoria', # Puntiamo alla nuova colonna universale
            values='COB', 
            aggfunc='sum',     
            margins=True,      
            margins_name='TOTALE' 
        ).reset_index()
        
        # Pulizia estetica (rimuove il nome tecnico dell'asse colonne)
        df_pivot.columns.name = None 
    else:
        # Fallback di sicurezza in caso di grafico vuoto
        df_pivot = pd.DataFrame(columns=['Anno', 'Categoria', 'COB'])
    
    # --- 4. LA LUCIDATURA ESTETICA (Rimozione .0 e NaN) ---
    def pulisci_celle(val):
        if pd.isna(val):                  # Se la cella è vuota (NaN)
            return ''                     # Sostituisci con stringa vuota
        if isinstance(val, (int, float)): # Se è un numero (es. 234.0)
            return str(int(val))          # Rimuovi il decimale e rendilo testo ('234')
        return str(val)                   # Lascia intatte le stringhe ('TOTALE')

    # Passiamo il panno lucidante su tutte le colonne
    for col in df_pivot.columns:
        df_pivot[col] = df_pivot[col].apply(pulisci_celle)

    return df_pivot

In [6]:
percorso_file_html="..\\esportazioni_quarto\\grafico_7_COB_NAZ_tutti_contratti_1525_tutte_le_naz.html"

df_risultato = estrai_tabella_da_html(percorso_file_html)
print(df_risultato)

      Anno AFGHANISTAN ALBANIA ALGERIA ANGOLA ANTIGUA E BARBUDA  \
0     2015           6    1493      16                            
1     2016          18    1424      18      1                     
2     2017          43    1601      24      7                     
3     2018          21    1780      35      6                 2   
4     2019          50    2001      28     10                     
5     2020          43    1773      20      7                     
6     2021          37    1928      33     11                     
7     2022          47    2026      52      8                 6   
8     2023          31    1693      49     11                 7   
9     2024          33    1670      33     15                 9   
10    2025          46    1634      37     10                 5   
11  TOTALE         375   19023     345     86                29   

   ARABIA SAUDITA ARGENTINA ARMENIA AUSTRALIA  ... UNGHERIA  \
0               5       168       6        23  ...      103   
1 

In [7]:
import re
import json
import pandas as pd

def estrai_tabella_da_html_con_titolo(percorso_file_html):
    """
    Legge un file HTML di Plotly, estrae la serie di dati e il TITOLO reale,
    restituendo un DataFrame Pandas (Crosstab) pulito (senza .0 o NaN) e una stringa.
    """
    with open(percorso_file_html, 'r', encoding='utf-8') as file:
        html_content = file.read()

    # 1. Regex Robusta: Catturiamo tutto ciò che sta dentro Plotly.newPlot( ... )
    pattern = r'Plotly\.newPlot\(\s*"[^"]+",\s*(\[.*?\])\s*,\s*(\{.*?\})\s*,\s*\{"responsive"'
    match = re.search(pattern, html_content, re.DOTALL)

    if not match:
        raise ValueError("Impossibile trovare il blocco dati Plotly nell'HTML.")

    json_data_str = match.group(1)
    json_layout_str = match.group(2)

    # 2. Convertiamo da Testo a Dizionari/Liste Python
    traces = json.loads(json_data_str)
    layout = json.loads(json_layout_str)

    # 3. ESTRAZIONE DEL TITOLO REALE
    titolo_estratto = "Tabella Dati" # Default
    if 'title' in layout and 'text' in layout['title']:
        titolo_estratto = layout['title']['text']
    elif 'text' in layout.get('title', {}):
        titolo_estratto = layout['title']['text']

    # 4. ESTRAZIONE DATI
    records = []
    for trace in traces:
        nome_categoria = trace.get('name', 'Sconosciuto')
        anni = trace.get('x', [])
        valori = trace.get('y', [])
        
        for anno, valore in zip(anni, valori):
            records.append({
                'Anno': anno,
                'Categoria': nome_categoria,
                'COB': valore
            })

    df_grezzo = pd.DataFrame(records)

    # 5. Creazione Tabella Pivot con Totali
    if not df_grezzo.empty:
        df_pivot = pd.pivot_table(
            df_grezzo, 
            index='Anno', 
            columns='Categoria', 
            values='COB', 
            aggfunc='sum',     
            margins=True,      
            margins_name='TOTALE' 
        ).reset_index()
        
        # Pulizia intestazioni
        df_pivot.columns.name = None 
    else:
        # Fallback se non ci sono dati
        df_pivot = pd.DataFrame(columns=['Anno', 'Categoria', 'COB'])

    # --- 6. LA LUCIDATURA ESTETICA (Rimozione .0 e NaN) ---
    def pulisci_celle(val):
        if pd.isna(val):                  # Se la cella è vuota (NaN)
            return ''                     # Sostituisci con stringa vuota
        if isinstance(val, (int, float)): # Se è un numero (es. 234.0)
            return str(int(val))          # Rimuovi il decimale e rendilo testo ('234')
        return str(val)                   # Lascia intatte le stringhe ('TOTALE')

    # Passiamo il panno lucidante su tutte le colonne della tabella
    for col in df_pivot.columns:
        df_pivot[col] = df_pivot[col].apply(pulisci_celle)

    # Restituiamo ENTRAMBI gli elementi finali
    return df_pivot, titolo_estratto

In [8]:
percorso_file_html="..\\esportazioni_quarto\\grafico_7_COB_NAZ_tutti_contratti_1525_tutte_le_naz.html"

df_tabella, titolo_dinamico = estrai_tabella_da_html_con_titolo(percorso_file_html)
print(titolo_dinamico)
print(df_risultato)

Trend Nazionalità: Tutti i contratti
      Anno AFGHANISTAN ALBANIA ALGERIA ANGOLA ANTIGUA E BARBUDA  \
0     2015           6    1493      16                            
1     2016          18    1424      18      1                     
2     2017          43    1601      24      7                     
3     2018          21    1780      35      6                 2   
4     2019          50    2001      28     10                     
5     2020          43    1773      20      7                     
6     2021          37    1928      33     11                     
7     2022          47    2026      52      8                 6   
8     2023          31    1693      49     11                 7   
9     2024          33    1670      33     15                 9   
10    2025          46    1634      37     10                 5   
11  TOTALE         375   19023     345     86                29   

   ARABIA SAUDITA ARGENTINA ARMENIA AUSTRALIA  ... UNGHERIA  \
0               5       168 

In [ ]:
##############################################
versione definitiva con trasponi righe colonne
##############################################

In [14]:
def estrai_tabella_da_html_con_titolo(percorso_file_html, trasponi=False):
    """
    Legge un file HTML di Plotly, estrae la serie di dati e il TITOLO reale,
    restituendo un DataFrame Pandas (Crosstab) pulito (senza .0 o NaN) e una stringa.
    Supporta la trasposizione assi tramite il parametro 'trasponi'.
    """
    with open(percorso_file_html, 'r', encoding='utf-8') as file:
        html_content = file.read()

    # 1. Regex Robusta: Catturiamo tutto ciò che sta dentro Plotly.newPlot( ... )
    pattern = r'Plotly\.newPlot\(\s*"[^"]+",\s*(\[.*?\])\s*,\s*(\{.*?\})\s*,\s*\{"responsive"'
    match = re.search(pattern, html_content, re.DOTALL)

    if not match:
        raise ValueError("Impossibile trovare il blocco dati Plotly nell'HTML.")

    json_data_str = match.group(1)
    json_layout_str = match.group(2)

    # 2. Convertiamo da Testo a Dizionari/Liste Python
    traces = json.loads(json_data_str)
    layout = json.loads(json_layout_str)

    # 3. ESTRAZIONE DEL TITOLO REALE
    titolo_estratto = "Tabella Dati" # Default
    if 'title' in layout and 'text' in layout['title']:
        titolo_estratto = layout['title']['text']
    elif 'text' in layout.get('title', {}):
        titolo_estratto = layout['title']['text']

    # 4. ESTRAZIONE DATI
    records = []
    for trace in traces:
        nome_categoria = trace.get('name', 'Sconosciuto')
        anni = trace.get('x', [])
        valori = trace.get('y', [])
        
        for anno, valore in zip(anni, valori):
            records.append({
                'Anno': anno,
                'Categoria': nome_categoria,
                'COB': valore
            })

    df_grezzo = pd.DataFrame(records)

    # 5. Creazione Tabella Pivot Dinamica (con opzione TRASPONI)
    if not df_grezzo.empty:
        # --- LA MAGIA DELLA TRASPOSIZIONE ---
        # Se trasponi=True, le categorie diventano le righe e gli anni le colonne.
        riga_index = 'Categoria' if trasponi else 'Anno'
        colonna_index = 'Anno' if trasponi else 'Categoria'

        df_pivot = pd.pivot_table(
            df_grezzo, 
            index=riga_index, 
            columns=colonna_index, 
            values='COB', 
            aggfunc='sum',     
            margins=True,      
            margins_name='TOTALE' 
        ).reset_index()
        
        # Pulizia intestazioni
        df_pivot.columns.name = None 
    else:
        # Fallback se non ci sono dati
        colonne_vuote = ['Categoria', 'Anno', 'COB'] if trasponi else ['Anno', 'Categoria', 'COB']
        df_pivot = pd.DataFrame(columns=colonne_vuote)

    # --- 6. LA LUCIDATURA ESTETICA (Rimozione .0 e NaN) ---
    def pulisci_celle(val):
        if pd.isna(val):                  # Se la cella è vuota (NaN)
            return ''                     # Sostituisci con stringa vuota
        if isinstance(val, (int, float)): # Se è un numero (es. 234.0)
            return str(int(val))          # Rimuovi il decimale e rendilo testo ('234')
        return str(val)                   # Lascia intatte le stringhe ('TOTALE')

    # Passiamo il panno lucidante su tutte le colonne della tabella
    for col in df_pivot.columns:
        df_pivot[col] = df_pivot[col].apply(pulisci_celle)

    # Restituiamo ENTRAMBI gli elementi finali
    return df_pivot, titolo_estratto

In [18]:
percorso_file_html="..\\esportazioni_quarto\\grafico_7_COB_NAZ_tutti_contratti_1525_tutte_le_naz.html"

df_tabella, titolo_dinamico = estrai_tabella_da_html_con_titolo(percorso_file_html, trasponi=False)
print(titolo_dinamico)
print(df_tabella)

Trend Nazionalità: Tutti i contratti
      Anno AFGHANISTAN ALBANIA ALGERIA ANGOLA ANTIGUA E BARBUDA  \
0     2015           6    1493      16                            
1     2016          18    1424      18      1                     
2     2017          43    1601      24      7                     
3     2018          21    1780      35      6                 2   
4     2019          50    2001      28     10                     
5     2020          43    1773      20      7                     
6     2021          37    1928      33     11                     
7     2022          47    2026      52      8                 6   
8     2023          31    1693      49     11                 7   
9     2024          33    1670      33     15                 9   
10    2025          46    1634      37     10                 5   
11  TOTALE         375   19023     345     86                29   

   ARABIA SAUDITA ARGENTINA ARMENIA AUSTRALIA  ... UNGHERIA  \
0               5       168 

In [ ]:
################################################
estrai tabella per grafici a barre
################################################

In [1]:
import re
import json
import pandas as pd

def estrai_tabella_barre_da_html_con_titolo(percorso_file_html):
    """
    Legge un file HTML di Plotly (grafico a barre), estrae le categorie, 
    i valori e (se presenti) le percentuali, restituendo un DataFrame Pandas pulito.
    """
    with open(percorso_file_html, 'r', encoding='utf-8') as file:
        html_content = file.read()

    # 1. Regex per catturare il blocco dati Plotly
    pattern = r'Plotly\.newPlot\(\s*"[^"]+",\s*(\[.*?\])\s*,\s*(\{.*?\})\s*,\s*\{"responsive"'
    match = re.search(pattern, html_content, re.DOTALL)

    if not match:
        raise ValueError("Impossibile trovare il blocco dati Plotly nell'HTML.")

    json_data_str = match.group(1)
    json_layout_str = match.group(2)

    traces = json.loads(json_data_str)
    layout = json.loads(json_layout_str)

    # 2. Estrazione Titolo (con fallback se non presente)
    titolo_estratto = "Tabella Distribuzione"
    if 'title' in layout and 'text' in layout['title']:
        titolo_estratto = layout['title']['text']
    elif 'text' in layout.get('title', {}):
        titolo_estratto = layout['title']['text']

    # 3. Estrazione Dati
    records = []
    
    # Nei grafici a barre di solito c'è una sola traccia principale
    for trace in traces:
        if trace.get('type') == 'bar':
            
            # Capiamo l'orientamento: se è orizzontale ('h'), le categorie sono la Y.
            if trace.get('orientation') == 'h':
                categorie = trace.get('y', [])
                valori = trace.get('x', [])
            else:
                categorie = trace.get('x', [])
                valori = trace.get('y', [])
                
            testi = trace.get('text', []) # Contiene es: "553006 (53.1%)"
            usa_testi = len(testi) == len(valori)

            for i in range(len(valori)):
                record = {
                    'Categoria': categorie[i],
                    'COB': valori[i]
                }
                
                # Bonus: Estraiamo la percentuale pulita dalle parentesi
                if usa_testi:
                    match_perc = re.search(r'\((.*?%)\)', str(testi[i]))
                    if match_perc:
                        record['Incidenza'] = match_perc.group(1)
                        
                records.append(record)

    df_risultato = pd.DataFrame(records)

    # 4. LA LUCIDATURA ESTETICA (Rimozione .0 e NaN)
    def pulisci_celle(val):
        if pd.isna(val):                  
            return ''                     
        if isinstance(val, (int, float)): 
            return str(int(val))          
        return str(val)                   

    if not df_risultato.empty:
        for col in df_risultato.columns:
            df_risultato[col] = df_risultato[col].apply(pulisci_celle)

    return df_risultato, titolo_estratto

# -------------- Esempio di utilizzo --------------
# percorso_file = "output_esportazioni_quarto/grafico_barre.html"
# df_tabella, titolo = estrai_tabella_barre_da_html_con_titolo(percorso_file)
# print(titolo)
# print(df_tabella.to_markdown(index=False))

In [4]:
percorso_file_html="..\\esportazioni_quarto\\grafico_1_COB_tipologia_contratti_15-25a.html"

df_tabella, titolo_dinamico = estrai_tabella_barre_da_html_con_titolo(percorso_file_html)
print(titolo_dinamico)
print(df_tabella)

Tabella Distribuzione
                                      Categoria     COB Incidenza
0                    Lavoro a tempo determinato  553006     53.1%
1                  Lavoro a tempo indeterminato  147830     14.2%
2   Lavoro interinale (o a scopo di somminis...  123263     11.8%
3      Lavoro intermittente a tempo determinato   84154      8.1%
4   Lavoro a tempo determinato  per sostituz...   40319      3.9%
5   Apprendistato professionalizzante o cont...   29108      2.8%
6                                     Tirocinio   15547      1.5%
7        Lavoro domestico a tempo indeterminato   14916      1.4%
8              Lavoro autonomo nello spettacolo    9555      0.9%
9      Collaborazione coordinata e continuativa    7423      0.7%
10  Lavoro intermittente  a tempo indeterminato    5190      0.5%
11  Collaborazione occasionale sportiva ex a...    4673      0.4%
12         Lavoro domestico a tempo determinato    2852      0.3%
13  Lavoro o attività socialmente utile (LSU...    210

In [ ]:
#------------------------ con il totale

In [5]:
import re
import json
import pandas as pd

def estrai_tabella_barre_da_html_con_titolo(percorso_file_html):
    """
    Legge un file HTML di Plotly (grafico a barre), estrae le categorie, 
    i valori e (se presenti) le percentuali, aggiunge una riga di TOTALE,
    e restituisce un DataFrame Pandas pulito.
    """
    with open(percorso_file_html, 'r', encoding='utf-8') as file:
        html_content = file.read()

    # 1. Regex per catturare il blocco dati Plotly
    pattern = r'Plotly\.newPlot\(\s*"[^"]+",\s*(\[.*?\])\s*,\s*(\{.*?\})\s*,\s*\{"responsive"'
    match = re.search(pattern, html_content, re.DOTALL)

    if not match:
        raise ValueError("Impossibile trovare il blocco dati Plotly nell'HTML.")

    json_data_str = match.group(1)
    json_layout_str = match.group(2)

    traces = json.loads(json_data_str)
    layout = json.loads(json_layout_str)

    # 2. Estrazione Titolo (con fallback se non presente)
    titolo_estratto = "Tabella Distribuzione"
    if 'title' in layout and 'text' in layout['title']:
        titolo_estratto = layout['title']['text']
    elif 'text' in layout.get('title', {}):
        titolo_estratto = layout['title']['text']

    # 3. Estrazione Dati
    records = []
    
    # Nei grafici a barre di solito c'è una sola traccia principale
    for trace in traces:
        if trace.get('type') == 'bar':
            
            # Capiamo l'orientamento: se è orizzontale ('h'), le categorie sono la Y.
            if trace.get('orientation') == 'h':
                categorie = trace.get('y', [])
                valori = trace.get('x', [])
            else:
                categorie = trace.get('x', [])
                valori = trace.get('y', [])
                
            testi = trace.get('text', []) # Contiene es: "553006 (53.1%)"
            usa_testi = len(testi) == len(valori)

            for i in range(len(valori)):
                record = {
                    'Categoria': categorie[i],
                    'COB': valori[i]
                }
                
                # Bonus: Estraiamo la percentuale pulita dalle parentesi
                if usa_testi:
                    match_perc = re.search(r'\((.*?%)\)', str(testi[i]))
                    if match_perc:
                        record['Incidenza'] = match_perc.group(1)
                        
                records.append(record)

    df_risultato = pd.DataFrame(records)

    # --- 4. AGGIUNTA DELLA RIGA TOTALE ---
    if not df_risultato.empty:
        totale_cob = df_risultato['COB'].sum()
        
        riga_totale = {
            'Categoria': 'TOTALE',
            'COB': totale_cob
        }
        
        # Se c'è la colonna percentuale, impostiamo il totale al 100%
        if 'Incidenza' in df_risultato.columns:
            riga_totale['Incidenza'] = '100.0%'
            
        # Aggiungiamo la nuova riga alla fine del DataFrame
        df_risultato.loc[len(df_risultato)] = riga_totale

    # 5. LA LUCIDATURA ESTETICA (Rimozione .0 e NaN)
    def pulisci_celle(val):
        if pd.isna(val):                  
            return ''                     
        if isinstance(val, (int, float)): 
            return str(int(val))          
        return str(val)                   

    if not df_risultato.empty:
        for col in df_risultato.columns:
            df_risultato[col] = df_risultato[col].apply(pulisci_celle)

    return df_risultato, titolo_estratto

# -------------- Esempio di utilizzo --------------
# percorso_file = "output_esportazioni_quarto/grafico_barre.html"
# df_tabella, titolo = estrai_tabella_barre_da_html_con_titolo(percorso_file)
# print(titolo)
# print(df_tabella.to_markdown(index=False))

In [6]:
percorso_file_html="..\\esportazioni_quarto\\grafico_1_COB_tipologia_contratti_15-25a.html"

df_tabella, titolo_dinamico = estrai_tabella_barre_da_html_con_titolo(percorso_file_html)
print(titolo_dinamico)
print(df_tabella)

Top 25 Tipologie Contrattuali
                                      Categoria      COB Incidenza
0                    Lavoro a tempo determinato   553006     53.1%
1                  Lavoro a tempo indeterminato   147830     14.2%
2   Lavoro interinale (o a scopo di somminis...   123263     11.8%
3      Lavoro intermittente a tempo determinato    84154      8.1%
4   Lavoro a tempo determinato  per sostituz...    40319      3.9%
5   Apprendistato professionalizzante o cont...    29108      2.8%
6                                     Tirocinio    15547      1.5%
7        Lavoro domestico a tempo indeterminato    14916      1.4%
8              Lavoro autonomo nello spettacolo     9555      0.9%
9      Collaborazione coordinata e continuativa     7423      0.7%
10  Lavoro intermittente  a tempo indeterminato     5190      0.5%
11  Collaborazione occasionale sportiva ex a...     4673      0.4%
12         Lavoro domestico a tempo determinato     2852      0.3%
13  Lavoro o attività socialment